# 🤖 ChatbotPTCTPT — RAG Chatbot Giải Thích Tài Liệu

**Kiến trúc:** Document Loaders → Text Splitter → HuggingFace Embeddings → ChromaDB → Gemini LLM → RAG Chain

**Các bước:**
1. **Cell 1** — Cài thư viện & nạp biến môi trường  
2. **Cell 2** — Định nghĩa hàm xử lý & chia nhỏ tài liệu  
3. **Cell 3** — Xây dựng Vector Database (chỉ cần chạy 1 lần)  
4. **Cell 4** — Khởi động Chatbot (setup chain, retriever)  
5. **Cell 5** — Đặt câu hỏi ← **GÕ CÂU HỎI Ở ĐÂY**  

---
**Cài đặt dependencies (chạy 1 lần trong terminal):**
```bash
pip install langchain langchain-core langchain-community langchain-google-genai langchain-huggingface langchain-chroma chromadb pymupdf docx2txt unstructured python-dotenv sentence-transformers
```

In [ ]:
# ============================================================
# CELL 1: IMPORT THƯ VIỆN & NẠP BIẾN MÔI TRƯỜNG
# ============================================================
import os
import sys
import shutil
from pathlib import Path
from dotenv import load_dotenv

# Kiểm tra version langchain
try:
    import langchain
    print(f"📦 langchain version: {langchain.__version__}")
except Exception:
    print("⚠️  Không tìm thấy langchain. Chạy lệnh cài đặt ở trên.")

# Thư viện xử lý tài liệu
from langchain_community.document_loaders import (
    PyMuPDFLoader,
    Docx2txtLoader,
    UnstructuredExcelLoader,
    UnstructuredPowerPointLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Thư viện Vector Database & LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Nạp biến môi trường từ file .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise EnvironmentError(
        "❌ KHÔNG TÌM THẤY GOOGLE_API_KEY!\n"
        "   Tạo file .env trong cùng thư mục với notebook, thêm dòng:\n"
        "   GOOGLE_API_KEY=your_api_key_here"
    )

print("✅ GOOGLE_API_KEY đã nạp thành công.")
print("✅ Sẵn sàng chạy các bước tiếp theo!")

In [ ]:
# ============================================================
# CELL 2: HÀM XỬ LÝ & CHIA NHỎ TÀI LIỆU
# ============================================================

def get_document_loader(file_path: Path):
    """
    Chọn Loader phù hợp dựa trên đuôi file.
    Trả về (loader, has_page_metadata):
      - has_page_metadata=True  => loader trả về metadata 'page' (0-based), chỉ PDF
      - has_page_metadata=False => không có metadata 'page'
    """
    ext = file_path.suffix.lower()
    path_str = str(file_path)

    loader_map = {
        '.pdf':  (PyMuPDFLoader,                True),
        '.docx': (Docx2txtLoader,               False),
        '.xlsx': (UnstructuredExcelLoader,      False),
        '.xls':  (UnstructuredExcelLoader,      False),
        '.pptx': (UnstructuredPowerPointLoader, False),
        '.ppt':  (UnstructuredPowerPointLoader, False),
    }

    if ext not in loader_map:
        return None, False

    LoaderClass, has_page = loader_map[ext]
    try:
        return LoaderClass(path_str), has_page
    except Exception as e:
        print(f"  ⚠️  Không thể khởi tạo loader cho {file_path.name}: {e}")
        return None, False


def process_and_chunk_all_files(root_directory: str, base_domain: str) -> list:
    """
    Quét đệ quy toàn bộ thư mục, đọc các file tài liệu,
    chia nhỏ thành chunks và gán metadata chuẩn xác.
    """
    root_path = Path(root_directory)
    if not root_path.exists():
        raise FileNotFoundError(f"❌ Không tìm thấy thư mục: {root_directory}")

    print(f"📂 Đang quét thư mục: {root_directory} ...")

    SUPPORTED_EXTENSIONS = {'.pdf', '.docx', '.xls', '.xlsx', '.ppt', '.pptx'}
    target_files = [f for f in root_path.rglob('*') if f.suffix.lower() in SUPPORTED_EXTENSIONS]

    if not target_files:
        print("⚠️  Không tìm thấy file tài liệu nào được hỗ trợ.")
        return []

    print(f"📄 Tìm thấy {len(target_files)} file. Bắt đầu xử lý...\n")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    all_chunks  = []
    success_count = 0
    skip_count    = 0

    for file_path in target_files:
        print(f"  📖 Đang đọc: {file_path.name}")

        loader, has_page_metadata = get_document_loader(file_path)
        if loader is None:
            print(f"     ⏭️  Bỏ qua (không có loader phù hợp)")
            skip_count += 1
            continue

        try:
            pages = loader.load()
            if not pages:
                print(f"     ⚠️  File rỗng hoặc không đọc được nội dung")
                skip_count += 1
                continue

            chunks    = text_splitter.split_documents(pages)
            doc_name  = file_path.stem
            file_type = file_path.suffix.lower()

            for chunk in chunks:
                if has_page_metadata:
                    page_num   = chunk.metadata.get('page', 0) + 1
                    source_url = f"{base_domain}/{file_path.name}#page={page_num}"
                else:
                    source_url = f"{base_domain}/{file_path.name}"

                chunk.metadata['document_name'] = doc_name
                chunk.metadata['source_url']    = source_url
                chunk.metadata['file_type']     = file_type

            all_chunks.extend(chunks)
            success_count += 1
            print(f"     ✅ {len(chunks)} chunks")

        except Exception as e:
            print(f"     ❌ LỖI: {e}")
            skip_count += 1

    print(f"\n{'='*50}")
    print(f"✅ HOÀN TẤT BƯỚC 1")
    print(f"   Thành công : {success_count}/{len(target_files)} file")
    print(f"   Bỏ qua/Lỗi: {skip_count}/{len(target_files)} file")
    print(f"   Tổng chunks: {len(all_chunks)}")
    print(f"{'='*50}")

    return all_chunks


print("✅ Đã định nghĩa các hàm xử lý tài liệu.")

In [ ]:
# ============================================================
# CELL 3: XÂY DỰNG VECTOR DATABASE
# ⚠️  CHỈ CẦN CHẠY 1 LẦN. Nếu chạy lại sẽ hỏi xác nhận.
# ============================================================

# --- CẤU HÌNH — Chỉnh sửa tại đây ---
THU_MUC_TAI_LIEU = r"D:\02_Work\Projects\ChatbotPTCTPT"
DOMAIN_WEB       = "https://website-cua-ban.com/vanban"
THU_MUC_DB       = "./chroma_db"
# ------------------------------------


def build_vector_database(chunks: list, persist_directory: str, force_rebuild: bool = False):
    """
    Nhúng vector bằng HuggingFace (cục bộ, miễn phí) và lưu vào ChromaDB.
    Xử lý theo batch để tránh tràn bộ nhớ với dataset lớn.
    """
    if not chunks:
        print("⚠️  Không có chunks. Hủy quá trình tạo Database.")
        return None

    db_path = Path(persist_directory)

    if db_path.exists() and any(db_path.iterdir()):
        if not force_rebuild:
            print(f"⚠️  Database đã tồn tại tại '{persist_directory}'.")
            confirm = input("Nhập 'yes' để xóa và tạo lại, hoặc Enter để bỏ qua: ").strip().lower()
            if confirm != 'yes':
                print("⏭️  Bỏ qua. Dùng database hiện có.")
                return None
        print(f"🗑️  Đang xóa database cũ tại '{persist_directory}'...")
        shutil.rmtree(persist_directory)

    print(f"\n🔄 Bắt đầu nhúng vector cho {len(chunks)} chunks...")
    print("📦 Đang tải mô hình embedding (chỉ tải 1 lần đầu tiên)...")

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("✅ Tải mô hình xong!")

    BATCH_SIZE    = 500
    vectorstore   = None

    for i in range(0, len(chunks), BATCH_SIZE):
        batch         = chunks[i : i + BATCH_SIZE]
        batch_num     = i // BATCH_SIZE + 1
        total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE
        print(f"  ⚙️  Batch {batch_num}/{total_batches} ({len(batch)} chunks)...")

        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                documents=batch,
                embedding=embeddings,
                persist_directory=persist_directory,
            )
        else:
            vectorstore.add_documents(batch)

    count = vectorstore._collection.count()
    print(f"\n✅ HOÀN TẤT! Database đã lưu tại: {persist_directory}")
    print(f"   Tổng số vectors: {count}")
    return vectorstore


# --- CHẠY ---
document_chunks = process_and_chunk_all_files(THU_MUC_TAI_LIEU, DOMAIN_WEB)

if document_chunks:
    print(f"\n🔍 Xem thử chunk đầu tiên ({document_chunks[0].metadata.get('file_type')}):")
    print(f"   [{document_chunks[0].metadata.get('document_name')}]")
    print(f"   {document_chunks[0].page_content[:200]}...")
    print()
    build_vector_database(document_chunks, THU_MUC_DB)
else:
    print("⚠️  Không có chunks nào. Kiểm tra lại thư mục tài liệu.")

In [ ]:
# ============================================================
# CELL 4: KHỞI ĐỘNG CHATBOT (setup chain)
# Chạy 1 lần sau khi có database. Sau đó dùng Cell 5 để hỏi.
# ============================================================

THU_MUC_DB = "./chroma_db"

# --- KIỂM TRA DATABASE ---
db_path = Path(THU_MUC_DB)
if not db_path.exists() or not any(db_path.iterdir()):
    raise FileNotFoundError(
        f"❌ Không tìm thấy database tại '{THU_MUC_DB}'.\n"
        "   Hãy chạy Cell 3 để xây dựng Vector Database trước."
    )

# 1. KẾT NỐI DATABASE
print("🔌 Đang kết nối với Vector Database...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vectorstore = Chroma(persist_directory=THU_MUC_DB, embedding_function=embeddings)
doc_count   = vectorstore._collection.count()
print(f"✅ Kết nối thành công! Database có {doc_count} vectors.")

if doc_count == 0:
    raise ValueError("❌ Database rỗng! Chạy lại Cell 3 để nạp dữ liệu.")

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# 2. GEMINI LLM
print("🧠 Đang khởi tạo Gemini LLM...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=api_key,
    temperature=0.1,
    max_output_tokens=2048,
)

# 3. SYSTEM PROMPT
SYSTEM_PROMPT = (
    "Bạn là một chuyên gia tư vấn về Phát triển Chương trình Giáo dục Phổ thông tại Việt Nam.\n"
    "Dưới đây là các tài liệu quy định (Context) được trích xuất từ hệ thống.\n"
    "Nhiệm vụ của bạn là trả lời câu hỏi DỰA HOÀN TOÀN VÀO CONTEXT NÀY.\n\n"
    "Nguyên tắc:\n"
    "1. Nếu thông tin KHÔNG có trong Context, trả lời thẳng: "
    "'Tôi không tìm thấy thông tin này trong các văn bản hiện hành.' — không tự suy diễn.\n"
    "2. Trình bày rõ ràng, dùng bullet point khi cần.\n"
    "3. Cuối mỗi ý quan trọng BẮT BUỘC đính kèm nguồn Markdown: [Tên văn bản](Link URL).\n\n"
    "--- NGỮ CẢNH (CONTEXT) ---\n"
    "{context}"
)

# 4. RAG CHAIN (LCEL — tương thích mọi version langchain)
def format_docs(docs):
    """Ghép nội dung các docs thành một chuỗi context có nhãn nguồn."""
    parts = []
    for doc in docs:
        name = doc.metadata.get('document_name', 'Không rõ')
        url  = doc.metadata.get('source_url', '')
        parts.append(f"[Nguồn: {name} | {url}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
])

rag_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(lambda x: format_docs(retriever.invoke(x["input"])))
    )
    | prompt
    | llm
    | StrOutputParser()
)

# Hàm hỏi dùng cho Cell 5
def hoi(cau_hoi: str, hien_nguon: bool = False):
    """
    Hàm hỏi chatbot. Gọi từ Cell 5.

    Args:
        cau_hoi   : Câu hỏi của bạn (string)
        hien_nguon: True = in thêm danh sách tài liệu đã tham khảo

    Ví dụ:
        hoi("Học sinh cần đáp ứng điều kiện gì để lên lớp?")
        hoi("Cấu trúc chương trình GDPT 2018?", hien_nguon=True)
    """
    print(f"👤 Câu hỏi: {cau_hoi}")
    print("🤖 Đang tìm kiếm và phân tích...\n")

    try:
        docs   = retriever.invoke(cau_hoi)
        answer = rag_chain.invoke({"input": cau_hoi})

        print("🤖 Trả lời:")
        print(answer)

        if hien_nguon and docs:
            print("\n📚 Tài liệu đã tham khảo:")
            seen = set()
            idx  = 1
            for doc in docs:
                url  = doc.metadata.get('source_url', 'N/A')
                name = doc.metadata.get('document_name', 'Không rõ')
                if url not in seen:
                    print(f"  {idx}. [{name}]({url})")
                    seen.add(url)
                    idx += 1
        return answer

    except Exception as e:
        print(f"❌ Lỗi: {e}")
        return None


print("\n✅ Chatbot đã sẵn sàng!")
print("="*60)
print("👉 Chuyển sang Cell 5 bên dưới để đặt câu hỏi.")
print("="*60)

In [ ]:
# ============================================================
# CELL 5: ĐẶT CÂU HỎI ← CHỈNH SỬA Ở ĐÂY
# Chạy Cell 1 → 4 trước, sau đó mỗi lần hỏi chỉ cần
# sửa câu hỏi rồi bấm Ctrl+Enter để chạy lại cell này.
# ============================================================

# ✏️  THAY ĐỔI CÂU HỎI TẠI ĐÂY
CAU_HOI = "Học sinh cần đáp ứng điều kiện gì để được lên lớp?"

# Đặt hien_nguon=True nếu muốn xem danh sách tài liệu đã dùng
hoi(CAU_HOI, hien_nguon=True)

In [ ]:
# ============================================================
# CELL 6 (TÙY CHỌN): CHẾ ĐỘ CHAT TƯƠNG TÁC
# Dùng khi chạy trên Jupyter Notebook / JupyterLab.
# Trên VS Code có thể ô nhập không hiện — dùng Cell 5 thay thế.
# ============================================================

print("💬 Chế độ chat tương tác")
print("   Gõ 'thoát' để kết thúc | Gõ 'nguon' để xem tài liệu tham khảo")
print("=" * 60)

last_docs = []

while True:
    print()
    try:
        user_input = input("👤 Bạn: ").strip()
    except EOFError:
        # VS Code đôi khi raise EOFError thay vì hiện ô nhập
        print("\nℹ️  VS Code không hỗ trợ input() tương tác.")
        print("   Hãy dùng Cell 5 (hàm hoi()) để đặt câu hỏi.")
        break

    if not user_input:
        continue

    if user_input.lower() in ['quit', 'thoát', 'exit']:
        print("👋 Tạm biệt!")
        break

    if user_input.lower() == 'nguon':
        if not last_docs:
            print("ℹ️  Chưa có câu trả lời nào.")
        else:
            print("\n📚 TÀI LIỆU ĐÃ THAM KHẢO:")
            seen = set()
            idx  = 1
            for doc in last_docs:
                url  = doc.metadata.get('source_url', 'N/A')
                name = doc.metadata.get('document_name', 'Không rõ')
                if url not in seen:
                    print(f"  {idx}. [{name}]({url})")
                    seen.add(url)
                    idx += 1
        continue

    print("🤖 Đang tìm kiếm và phân tích...\n")
    try:
        last_docs = retriever.invoke(user_input)
        answer    = rag_chain.invoke({"input": user_input})
        print(f"🤖 Chatbot:\n{answer}")
    except Exception as e:
        print(f"❌ Lỗi: {e}")